In [1]:
!pip install langchain langchain-core langchain-community langchain-text-splitters langchain-huggingface faiss-cpu pypdf sentence-transformers

In [2]:
!pip install python-dotenv langchain-groq langchain-google-genai

In [3]:
import os
from dotenv import load_dotenv

load_dotenv("Tokens.env")

for name in ["GROQ_API_KEY", "GEMINI_API_KEY", "OPENROUTER_API_KEY"]:
    print(name, "->", "found" if os.getenv(name) else "missing")

python-dotenv could not parse statement starting at line 3
python-dotenv could not parse statement starting at line 8
python-dotenv could not parse statement starting at line 12


GROQ_API_KEY -> found
GEMINI_API_KEY -> found
OPENROUTER_API_KEY -> missing


In [4]:
#<<<<<<<<<<<<<<<<<<<<<--CONFIG CELL-->>>>>>>>>>>>>>>>>>#

DOCS_DIR = "data/documents"
INDEX_DIR = "faiss_index"

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
GROQ_MODEL = "openai/gpt-oss-20b"
GEMINI_MODEL = "gemini-3.8-flash"

CHUNK_SIZE = 800
CHUNK_OVERLAP = 150
TOP_K = 4

In [5]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader

pdf_path = next(Path(DOCS_DIR).glob("*.pdf"))
docs = PyPDFLoader(str(pdf_path)).load()
print(f"Loaded {len(docs)} pages from {pdf_path.name}")

C:\Users\samee\AppData\Local\Temp\ipykernel_20408\1941925653.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Loaded 6 pages from Nova_Tech_Employee_Handbook.pdf


In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)
chunks = splitter.split_documents(docs)

print("Total chunks:", len(chunks))

Total chunks: 13


In [7]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)
vectorstore = FAISS.from_documents(chunks, embeddings)
print("Index built with", vectorstore.index.ntotal, "vectors")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Index built with 13 vectors


In [8]:
vectorstore.save_local(INDEX_DIR)
vectorstore = FAISS.load_local(
    INDEX_DIR, embeddings, allow_dangerous_deserialization=True
)

In [9]:
retriever = vectorstore.as_retriever(search_kwargs={"k": TOP_K})

In [10]:
from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI

def build_llm():
    models = []
    if os.getenv("GROQ_API_KEY"):
        models.append(ChatGroq(
            model=GROQ_MODEL, temperature=0.2,
            api_key=os.getenv("GROQ_API_KEY"),
            timeout=30, max_retries=1))
    if os.getenv("GEMINI_API_KEY"):
        models.append(ChatGoogleGenerativeAI(
            model=GEMINI_MODEL, temperature=0.2,
            google_api_key=os.getenv("GEMINI_API_KEY"),
            timeout=30, max_retries=1))
    if not models:
        raise RuntimeError("No API keys found. Add a key to Tokens.env.")
    return models[0].with_fallbacks(models[1:])

llm = build_llm()
print(llm.invoke("Say hello in one short sentence.").text)

Hello!


In [11]:
results = vectorstore.similarity_search("How many sick leave days do I get?", k=1)
for i, doc in enumerate(results, 1):
    print(f"\n--- Chunk {i} | page {doc.metadata['page'] + 1} ---")
    print(doc.page_content[:300])


--- Chunk 1 | page 1 ---
Attendance is recorded using the biometric system in the office and the NovaHR mobile
app for remote days.
3. Leave Policy
Annual leave: Every full-time employee gets 18 paid leave days per year, credited at the
start of January. Of these, 12 are casual leave days and 6 are sick leave days.
Sick lea


In [12]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)

vectorstore = FAISS.load_local(
    INDEX_DIR,
    embeddings,
    allow_dangerous_deserialization=True,
)
print("Loaded vectors:", vectorstore.index.ntotal)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loaded vectors: 13


In [13]:
retriever = vectorstore.as_retriever(search_kwargs={"k": TOP_K})

In [14]:
def show(question):
    results = retriever.invoke(question)
    print("QUESTION:", question)
    for i, doc in enumerate(results, 1):
        page = doc.metadata["page"] + 1
        text = doc.page_content[:250].replace("\n", " ")
        print(f"  [{i}] page {page}: {text}")
    print()

show("How many sick leave days do I get?")

QUESTION: How many sick leave days do I get?
  [1] page 1: Attendance is recorded using the biometric system in the office and the NovaHR mobile app for remote days. 3. Leave Policy Annual leave: Every full-time employee gets 18 paid leave days per year, credited at the start of January. Of these, 12 are cas
  [2] page 2: Maternity leave: Female employees are entitled to 26 weeks of paid maternity leave. Paternity leave: Male employees are entitled to 10 working days of paid paternity leave, to be taken within 3 months of the child's birth. Public holidays: The compan
  [3] page 1: question not covered here, employees should contact the HR team at hr@novatech-example.com or dial extension 200. 2. Working Hours and Attendance The standard working week is Monday to Friday. Office hours are 9:30 AM to 6:30 PM, which includes a one
  [4] page 6: Referral bonus: Employees receive 30,000 INR for referring a candidate who is hired and completes 6 months with the company. Free lunch is provide

In [15]:
questions = [
    "What is the notice period after confirmation?",
    "How much is the hotel limit per night in metro cities?",
    "How many days of paternity leave are there?",
    "What is the password policy?",
    "How much is the learning budget?",
    "Can I do freelance work?",
    "What is the CEO's name?",
]

for q in questions:
    show(q)

QUESTION: What is the notice period after confirmation?
  [1] page 6: Referral bonus: Employees receive 30,000 INR for referring a candidate who is hired and completes 6 months with the company. Free lunch is provided on all office days at the cafeteria in the Hyderabad and Bengaluru offices. 10. Joining, Notice Period
  [2] page 1: question not covered here, employees should contact the HR team at hr@novatech-example.com or dial extension 200. 2. Working Hours and Attendance The standard working week is Monday to Friday. Office hours are 9:30 AM to 6:30 PM, which includes a one
  [3] page 1: Attendance is recorded using the biometric system in the office and the NovaHR mobile app for remote days. 3. Leave Policy Annual leave: Every full-time employee gets 18 paid leave days per year, credited at the start of January. Of these, 12 are cas
  [4] page 2: Maternity leave: Female employees are entitled to 26 weeks of paid maternity leave. Paternity leave: Male employees are entitled to 10 

In [16]:
import re

tests = {
    "How many sick leave days do I get?": "sick leave days",
    "What is the notice period after confirmation?": "60 days",
    "How much is the hotel limit per night in metro cities?": "6,000 INR per night",
    "How many days of paternity leave are there?": "10 working days",
    "What is the password policy?": "12 characters",
    "How much is the learning budget?": "25,000",
    "Can I do freelance work?": "freelance",
}

def norm(text):
    return re.sub(r"\s+", " ", text).lower()

def check(retriever, tests):
    found = 0
    for question, keyword in tests.items():
        docs = retriever.invoke(question)
        rank = next((i for i, d in enumerate(docs, 1)
                     if keyword.lower() in norm(d.page_content)), None)
        print(f"rank {rank}" if rank else "MISS  ", "|", question)
        found += rank is not None
    print(f"\n{found}/{len(tests)} questions have the answer in the top {len(docs)}")

check(retriever, tests)

rank 1 | How many sick leave days do I get?
rank 1 | What is the notice period after confirmation?
rank 1 | How much is the hotel limit per night in metro cities?
rank 1 | How many days of paternity leave are there?
rank 1 | What is the password policy?
rank 1 | How much is the learning budget?
rank 1 | Can I do freelance work?

7/7 questions have the answer in the top 4


In [17]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

for size, overlap in [(400, 80), (800, 150), (1200, 200)]:
    splitter = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=overlap)
    test_chunks = splitter.split_documents(docs)
    test_store = FAISS.from_documents(test_chunks, embeddings)
    for k in [2, 4]:
        r = test_store.as_retriever(search_kwargs={"k": k})
        hits = sum(
            any(kw.lower() in norm(d.page_content) for d in r.invoke(q))
            for q, kw in tests.items()
        )
        print(f"chunk_size={size:>4}  k={k}  chunks={len(test_chunks):>3}  score={hits}/{len(tests)}")

chunk_size= 400  k=2  chunks= 25  score=7/7
chunk_size= 400  k=4  chunks= 25  score=7/7
chunk_size= 800  k=2  chunks= 13  score=7/7
chunk_size= 800  k=4  chunks= 13  score=7/7
chunk_size=1200  k=2  chunks= 11  score=6/7
chunk_size=1200  k=4  chunks= 11  score=7/7


In [18]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a helpful HR assistant for Nova Tech Solutions. "
     "Answer the question using ONLY the context below. "
     "If the answer is not in the context, say: \"I couldn't find that in the handbook.\" "
     "Do not make things up.\n\nContext:\n{context}"),
    ("human", "{question}"),
])

In [19]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [20]:
questions = [
    "How many sick leave days do I get?",
    "What is the notice period after confirmation?",
    "What is the CEO's name?",
]

for q in questions:
    print("Q:", q)
    print("A:", rag_chain.invoke(q))
    print()

Q: How many sick leave days do I get?
A: You are entitled to **6 sick leave days** per year.

Q: What is the notice period after confirmation?
A: After confirmation, the notice period is 60 days for all employees up to Senior Engineer level and 90 days for Lead and above.

Q: What is the CEO's name?
A: I couldn't find that in the handbook.



In [21]:
%%writefile ChatBot.py
import hashlib
import os
import tempfile
import time

import streamlit as st
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ---------------------------------------------------------------- settings
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
GROQ_MODEL = "openai/gpt-oss-20b"
GEMINI_MODEL = "gemini-3.8-flash"
TOP_K = 4
CHUNK_SIZE = 800
CHUNK_OVERLAP = 150
MAX_PAGES = 150
REFUSAL = "i couldn't find that in the document"

# Models are tried in this order. Add a spare key to Tokens.env (or the app
# secrets) and it is picked up automatically - no code change needed.
KEY_SLOTS = [
    ("Groq", "GROQ_API_KEY"),
    ("Gemini", "GEMINI_API_KEY"),
    ("Groq (spare)", "GROQ_API_KEY_2"),
    ("Gemini (spare)", "GEMINI_API_KEY_2"),
]

SAMPLE_QUESTIONS = [
    ("📝", "Summarize this document in five bullet points."),
    ("📌", "What are the key rules or policies mentioned?"),
    ("📅", "List any dates, deadlines or amounts mentioned."),
]

STEPS = [
    ("📤", "Upload", "Add one or more PDF files in the sidebar."),
    ("✂️", "Chunk", "The app reads every page and splits the text into small, overlapping chunks."),
    ("🧮", "Embed", "Each chunk is turned into numbers (an embedding) and stored in a FAISS search index."),
    ("🔎", "Retrieve", "When you ask a question, the app finds the chunks that match it best."),
    ("🤖", "Answer", "An AI model reads only those chunks and writes the answer. Sources are shown."),
    ("⏳", "Be patient",
     "Answers usually appear in a few seconds, but can take 1-2 minutes when "
     "the document is large or the AI service is busy."),
]

AVATARS = {"user": "🙋", "assistant": "🤖"}

PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You are a helpful assistant that answers questions about a document. "
     "Answer the question using ONLY the context below. "
     "If the answer is not in the context, say: \"I couldn't find that in the document.\" "
     "Do not make things up.\n\nContext:\n{context}"),
    ("human", "{question}"),
])

st.set_page_config(
    page_title="Document Q&A Chatbot",
    page_icon="📄",
    layout="wide",
)

load_dotenv("Tokens.env")


# ---------------------------------------------------------------- helpers
def get_key(name):
    """Read a key from Streamlit secrets (deployment) or Tokens.env (local)."""
    try:
        if name in st.secrets:
            return st.secrets[name]
    except Exception:
        pass
    return os.getenv(name)


@st.cache_resource(show_spinner="Loading the embedding model...")
def load_embeddings():
    return HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)


@st.cache_resource(show_spinner=False)
def load_models():
    """Return [(label, model), ...] for every key that is available."""
    models = []
    for label, secret_name in KEY_SLOTS:
        key = get_key(secret_name)
        if not key:
            continue
        if label.startswith("Groq"):
            model = ChatGroq(
                model=GROQ_MODEL, temperature=0.2, api_key=key,
                timeout=30, max_retries=1)
        else:
            model = ChatGoogleGenerativeAI(
                model=GEMINI_MODEL, temperature=0.2, google_api_key=key,
                timeout=30, max_retries=1)
        models.append((label, model))
    if not models:
        raise RuntimeError(
            "No API keys found. Add GROQ_API_KEY and/or GEMINI_API_KEY "
            "to Tokens.env (local) or to the app secrets (deployment).")
    return models


def answer_question(context, question):
    """Try each model in turn. Returns (answer, label of the model that answered)."""
    last_error = None
    for label, model in MODELS:
        try:
            chain = PROMPT | model | RunnableLambda(lambda m: m.text)
            answer = chain.invoke({"context": context, "question": question})
            return answer, label
        except Exception as err:
            last_error = err
    raise last_error


def files_key(files):
    digest = hashlib.md5()
    for f in files:
        digest.update(f.name.encode())
        digest.update(f.getvalue())
    return digest.hexdigest()


def build_index(files, log):
    """Read the PDFs, split them into chunks and build a FAISS index."""
    docs = []
    file_info = []
    log("📖 Reading PDF pages...")
    for f in files:
        with tempfile.NamedTemporaryFile(delete=False, suffix=".pdf") as tmp:
            tmp.write(f.getvalue())
            tmp_path = tmp.name
        try:
            pages = PyPDFLoader(tmp_path).load()
        finally:
            os.remove(tmp_path)
        for page in pages:
            page.metadata["file"] = f.name
        docs.extend(pages)
        file_info.append({"name": f.name, "pages": len(pages)})

    if len(docs) > MAX_PAGES:
        raise ValueError(
            f"These files have {len(docs)} pages. Please upload at most "
            f"{MAX_PAGES} pages in total.")
    if not any(d.page_content.strip() for d in docs):
        raise ValueError(
            "No readable text was found. Scanned (image-only) PDFs are not supported.")

    log("✂️ Splitting the text into chunks...")
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
    chunks = splitter.split_documents(docs)

    log("🧮 Creating embeddings and the search index...")
    vectorstore = FAISS.from_documents(chunks, load_embeddings())

    stats = {
        "files": file_info,
        "pages": len(docs),
        "chunks": len(chunks),
        "words": sum(len(d.page_content.split()) for d in docs),
        "preview": [c.page_content[:300] for c in chunks[:3]],
    }
    return vectorstore, stats


def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)


def render_sources(sources):
    with st.expander(f"📚 Sources ({len(sources)})"):
        for i, s in enumerate(sources, 1):
            with st.container(border=True):
                st.caption(f"Source {i} · 📄 {s['file']} · page {s['page']}")
                st.write(s["text"])


def chat_as_text(messages):
    lines = ["# Chat export", ""]
    for m in messages:
        who = "You" if m["role"] == "user" else "Assistant"
        lines.append(f"**{who}:** {m['content']}")
        lines.append("")
    return "\n".join(lines)


def reset_document():
    st.session_state.index_key = None
    st.session_state.vectorstore = None
    st.session_state.stats = None
    st.session_state.messages = []


# ---------------------------------------------------------------- state
for key, default in {
    "messages": [], "pending": None,
    "index_key": None, "vectorstore": None, "stats": None,
}.items():
    st.session_state.setdefault(key, default)

try:
    MODELS = load_models()
except RuntimeError as err:
    st.error(f"🔑 {err}")
    st.stop()

# ---------------------------------------------------------------- sidebar
with st.sidebar:
    st.header("📁 Your document")
    uploaded = st.file_uploader(
        "Upload one or more PDF files",
        type=["pdf"],
        accept_multiple_files=True,
        help=f"Text-based PDFs only, up to {MAX_PAGES} pages in total.",
    )
    files = uploaded or []

    if not files:
        if st.session_state.index_key is not None:
            reset_document()
    else:
        new_key = files_key(files)
        if new_key != st.session_state.index_key:
            error = None
            result = None
            with st.status("⚙️ Processing your document...", expanded=True) as status:
                try:
                    result = build_index(files, status.write)
                except ValueError as err:
                    error = str(err)
                except Exception:
                    error = "Sorry, this file could not be read. Try another PDF."
                if error:
                    status.update(label="Could not process the document", state="error")
                else:
                    status.update(label="Document ready!", state="complete", expanded=False)
            if error:
                st.error(f"⚠️ {error}")
                st.stop()
            st.session_state.vectorstore, st.session_state.stats = result
            st.session_state.index_key = new_key
            st.session_state.messages = []
            st.toast("Document indexed. Ask away!", icon="✅")

    ready = st.session_state.vectorstore is not None

    if ready:
        s = st.session_state.stats
        st.success(f"✅ Ready: {len(s['files'])} file(s), {s['pages']} pages, {s['chunks']} chunks")
    else:
        st.info("👆 Upload a PDF to get started.")

    st.divider()
    st.subheader("💡 Try asking")
    for i, (icon, sample) in enumerate(SAMPLE_QUESTIONS):
        if st.button(f"{icon} {sample}", key=f"sample_{i}", disabled=not ready):
            st.session_state.pending = sample

    st.divider()
    st.subheader("⚙️ Settings")
    top_k = st.slider(
        "🔢 Chunks to retrieve", 2, 8, TOP_K,
        help="How many pieces of the document are given to the AI for each question.")
    show_sources = st.toggle("📚 Show sources", value=True)

    col_a, col_b = st.columns(2)
    with col_a:
        if st.button("🧹 Clear", disabled=not st.session_state.messages):
            st.session_state.messages = []
            st.rerun()
    with col_b:
        st.download_button(
            "⬇️ Save",
            data=chat_as_text(st.session_state.messages),
            file_name="chat.md",
            mime="text/markdown",
            disabled=not st.session_state.messages,
        )

    st.divider()
    st.caption("🔑 AI models ready: " + ", ".join(label for label, _ in MODELS))
    st.caption("Built with LangChain · FAISS · Streamlit")

# ---------------------------------------------------------------- main page
st.title("📄 Document Q&A Chatbot")
st.caption("Upload a PDF and chat with it. Answers come only from your document, with sources. ✨")

tab_chat, tab_how, tab_doc = st.tabs(["💬 Chat", "🧭 How it works", "📊 Document"])

# ---- chat tab: history
with tab_chat:
    if not st.session_state.messages:
        if ready:
            with st.chat_message("assistant", avatar=AVATARS["assistant"]):
                st.markdown(
                    "👋 **Hi!** Your document is ready. Ask me anything about it, "
                    "or pick a sample question from the sidebar.")
        else:
            st.info("👈 Upload a PDF in the sidebar to start chatting. "
                    "New here? Open the **🧭 How it works** tab.")

    for msg in st.session_state.messages:
        with st.chat_message(msg["role"], avatar=AVATARS[msg["role"]]):
            st.markdown(msg["content"])
            if show_sources and msg.get("sources"):
                render_sources(msg["sources"])
            if msg.get("elapsed"):
                st.caption(f"⏱️ {msg['elapsed']:.1f} s · 🤖 {msg['model']}")

# ---- new question (the chat box stays pinned at the bottom of the page)
question = st.chat_input("💬 Ask a question about your document...", disabled=not ready)
if st.session_state.pending:
    question = st.session_state.pending
    st.session_state.pending = None

if question and ready:
    st.session_state.messages.append({"role": "user", "content": question})
    with tab_chat:
        with st.chat_message("user", avatar=AVATARS["user"]):
            st.markdown(question)

        with st.chat_message("assistant", avatar=AVATARS["assistant"]):
            try:
                with st.spinner("🤔 Thinking... this can take a few seconds, "
                                "longer if the AI service is busy."):
                    start = time.time()
                    retriever = st.session_state.vectorstore.as_retriever(
                        search_kwargs={"k": top_k})
                    found = retriever.invoke(question)
                    answer, used = answer_question(format_docs(found), question)
                    elapsed = time.time() - start
            except Exception as err:
                st.error("😕 Sorry, I could not get an answer. The AI service may be "
                         "busy or out of free quota. Please try again in a minute.")
                st.caption(f"Error type: {type(err).__name__}")
            else:
                st.markdown(answer)
                sources = []
                if REFUSAL not in answer.lower():
                    sources = [
                        {
                            "file": d.metadata.get("file", "document"),
                            "page": d.metadata.get("page", 0) + 1,
                            "text": d.page_content[:400],
                        }
                        for d in found
                    ]
                    if show_sources:
                        render_sources(sources)
                st.caption(f"⏱️ {elapsed:.1f} s · 🤖 {used}")
                if used != MODELS[0][0]:
                    st.toast(f"The main model was busy, so {used} answered.", icon="🔁")
                st.session_state.messages.append({
                    "role": "assistant", "content": answer,
                    "sources": sources, "elapsed": elapsed, "model": used,
                })

# ---- how it works tab
with tab_how:
    st.subheader("How this app works")
    for start_index, row in ((1, STEPS[:3]), (4, STEPS[3:])):
        columns = st.columns(3)
        for n, (column, (icon, title, text)) in enumerate(zip(columns, row), start_index):
            with column.container(border=True):
                st.markdown(f"#### {n}. {icon} {title}")
                st.write(text)
    st.info("⏳ **Please be patient:** the first answer after an upload can take longer, "
            "and answers can take up to 1-2 minutes when the document is large or "
            "the AI service is busy.")

# ---- document tab (filled last so the counters are up to date)
with tab_doc:
    if not ready:
        st.info("📄 Upload a PDF to see its details here.")
    else:
        stats = st.session_state.stats
        m1, m2, m3, m4 = st.columns(4)
        m1.metric("📁 Files", len(stats["files"]))
        m2.metric("📄 Pages", stats["pages"])
        m3.metric("🧩 Chunks", stats["chunks"])
        m4.metric("📝 Words", f"{stats['words']:,}")

        st.subheader("Files")
        for info in stats["files"]:
            st.write(f"📄 **{info['name']}** · {info['pages']} pages")

        asked = sum(1 for m in st.session_state.messages if m["role"] == "user")
        times = [m["elapsed"] for m in st.session_state.messages if m.get("elapsed")]
        st.subheader("This session")
        q1, q2 = st.columns(2)
        q1.metric("💬 Questions asked", asked)
        q2.metric("⏱️ Average answer time", f"{sum(times) / len(times):.1f} s" if times else "-")

        with st.expander("🔍 Preview the first chunks"):
            for i, text in enumerate(stats["preview"], 1):
                with st.container(border=True):
                    st.caption(f"Chunk {i}")
                    st.write(text)

Overwriting ChatBot.py


In [22]:
import os
os.makedirs(".streamlit", exist_ok=True)

In [23]:
%%writefile .streamlit/config.toml
[theme]
base = "light"
primaryColor = "#4F46E5"
backgroundColor = "#FFFFFF"
secondaryBackgroundColor = "#F3F4F8"
textColor = "#1F2937"
baseRadius = "large"
font = "Inter:https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&display=swap"
headingFont = "Poppins:https://fonts.googleapis.com/css2?family=Poppins:wght@600;700&display=swap"

Overwriting .streamlit/config.toml


In [24]:
%%writefile requirements.txt
streamlit
langchain
langchain-core
langchain-community
langchain-text-splitters
langchain-huggingface
langchain-groq
langchain-google-genai
python-dotenv
faiss-cpu
pypdf
sentence-transformers
pytest

Overwriting requirements.txt


In [25]:
%%writefile .gitignore
venv/
.env
*.env
.streamlit/secrets.toml
__pycache__/
.ipynb_checkpoints/
faiss_index/

Overwriting .gitignore


In [26]:
import os
os.makedirs("tests", exist_ok=True)

In [34]:
%%writefile tests/test_retriever.py
import glob

import pytest
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

PDFS = [p for p in glob.glob("**/*.pdf", recursive=True) if "venv" not in p]


@pytest.mark.skipif(not PDFS, reason="no sample PDF found in the project")
def test_retriever_returns_chunks():
    pages = PyPDFLoader(PDFS[0]).load()
    chunks = RecursiveCharacterTextSplitter(
        chunk_size=800, chunk_overlap=150).split_documents(pages)
    store = FAISS.from_documents(
        chunks, HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2"))
    found = store.as_retriever(search_kwargs={"k": 4}).invoke("leave policy")
    assert len(found) == 4
    assert all(d.page_content.strip() for d in found)

Overwriting tests/test_retriever.py


In [35]:
!python -m pytest tests -q

.                                                                        [100%]
============================== warnings summary ===============================
tests\test_retriever.py:4
  C:\Users\samee\Downloads\Rag-ChatBot\tests\test_retriever.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
    from langchain_community.document_loaders import PyPDFLoader

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
1 passed, 1 warning in 29.41s


In [31]:
%%writefile README.md
# 📄 Document Q&A Chatbot (RAG)

Upload a PDF and ask questions about it. The chatbot answers **only from your document** and shows the sources it used.

🔗 **Live app:** _add your Streamlit link here after deployment_

## ✨ Features
- 📤 Upload one or more PDFs in the app (no document is hard-coded)
- 🔎 Retrieval-Augmented Generation: answers are grounded in your file
- 📚 Sources shown for every answer (file and page)
- 🤖 Hosted AI models with automatic backup: Groq first, then Gemini
- 🧭 "How it works" tab and a "Document" tab with live stats
- ⚙️ Adjustable number of retrieved chunks, chat download

## 🧠 How it works
1. **Upload** a PDF in the sidebar.
2. **Chunk:** the text is split into 800-character chunks with 150 overlap.
3. **Embed:** each chunk is converted to a vector (`all-MiniLM-L6-v2`, runs locally) and stored in a FAISS index.
4. **Retrieve:** your question is matched to the top-k most similar chunks.
5. **Answer:** an LLM reads only those chunks and writes the answer.
   If the answer is not in the document, it says so.
6. **Be patient:** answers usually take a few seconds, but can take around 80–120 seconds
   when the document is large or the AI service is busy.

## 🛠 Tech stack
LangChain · FAISS · Sentence-Transformers · Groq (`openai/gpt-oss-20b`) · Google Gemini · Streamlit

## 🚀 Run locally
```bash
pip install -r requirements.txt
```
Create a file named `Tokens.env` (see `.env.example`) with your own free API keys:
```
GROQ_API_KEY=...
GEMINI_API_KEY=...
```
Then start the app:
```bash
python -m streamlit run ChatBot.py
```

## 🔑 API keys and backup
The app tries the models in order: Groq → Gemini → spare keys
(`GROQ_API_KEY_2`, `GEMINI_API_KEY_2`). If one key hits its free limit or expires,
the next one answers automatically. Keys are never stored in the code or in GitHub.

## ☁️ Deployment
Deployed on Streamlit Community Cloud. Keys are added in the app's **Secrets** settings.

## ⚠️ Limitations
- Works with text-based PDFs only (scanned images are not supported)
- Up to 150 pages in total per session
- Free API plans have rate limits

## 📁 Project files
| File | Purpose |
|---|---|
| `ChatBot.py` | Streamlit app |
| `rag_chatbot.ipynb` | Notebook: step-by-step build and experiments |
| `.streamlit/config.toml` | Fonts and theme |
| `requirements.txt` | Dependencies |
| `tests/` | Retriever test |
| `.env.example` | Template for API keys |

Overwriting README.md


In [37]:
import re, os
pattern = re.compile(r"gsk_[A-Za-z0-9]{10,}")
for root, dirs, files in os.walk("."):
    dirs[:] = [d for d in dirs if d not in ("venv", ".git")]
    for name in files:
        if name == "Tokens.env":
            continue
        path = os.path.join(root, name)
        try:
            for n, line in enumerate(open(path, encoding="utf-8", errors="ignore"), 1):
                if pattern.search(line):
                    print(path, "line", n)
        except OSError:
            pass

.\.env.example line 3
.\.env.example line 6
.\.ipynb_checkpoints\Tokens-checkpoint.env line 1
